# Data Reading and Consolidation — Datastream Exports

Converts the Excel block exports (`MKT_M_*`, `MKT_W_*`, `MKT_A_*`) into consolidated
long-format tables, saved as Parquet for fast reloading in later stages.

## Outputs written to the data folder

| File | Content |
|---|---|
| `raw_monthly.parquet`   | monthly market and I/B/E/S series |
| `raw_weekly.parquet`    | weekly total return index |
| `raw_annual.parquet`    | annual Worldscope accounting items |
| `fiscal_year_end.parquet` | fiscal period end dates (WC05350) |
| `security_meta.parquet` | security names, dead/active status, delisting dates |
| `read_log.csv`          | per-file reading outcome |
| `raw_riskfree.parquet`  | 3-month EURIBOR (if `RF_M.xlsx` present) |
| `raw_index.parquet`     | market indices (if `IDX_W.xlsx` present) |

## Scope

Data are read **as retrieved**. Post-delisting padding removal, deduplication,
the accounting lag and characteristic construction are handled in subsequent steps.

## Parsing notes

Three aspects of the exports require explicit handling:

1. **`NA` strings and `#ERROR` columns.** Failed retrievals are counted and dropped
   rather than silently coerced.
2. **`WC05350` holds dates, not numbers.** Converting it with the numeric series
   would turn each fiscal year-end into a nanosecond timestamp; it is therefore
   routed to a separate output.
3. **Metadata must be aggregated across files.** A security appears in three files;
   market series carry the `DEAD` marker and the delisting date in their name, while
   Worldscope columns carry the accounting item name instead. Taking the first row
   encountered would misclassify delisted securities as active, so the aggregation
   takes the union of the evidence.

## 1. Setup

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os, glob
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Thesis/data"

files = sorted(glob.glob(os.path.join(DATA_DIR, "MKT_*.xlsx")))
print(f"Block files found: {len(files)}")
for fam, label in [('M', 'monthly'), ('W', 'weekly'), ('A', 'annual')]:
    n = len([f for f in files if f"_{fam}_" in os.path.basename(f)])
    print(f"  {label:9s}: {n:>3}")

extra = sorted(glob.glob(os.path.join(DATA_DIR, "RF_*.xlsx")) +
               glob.glob(os.path.join(DATA_DIR, "IDX_*.xlsx")))
print(f"\nStandalone series: {[os.path.basename(f) for f in extra] or 'none found'}")

Block files found: 156
  monthly  :  52
  weekly   :  52
  annual   :  52

Standalone series: ['IDX_W.xlsx', 'RF_M.xlsx']


## 2. Parser

`parse_ds_file` locates the `Code` header row, splits each column heading into a
`SYMBOL(DATATYPE)` pair, and stacks the values into long format.
`aggregate_meta` collapses the per-column metadata to one row per security.

In [15]:
import openpyxl, re, os, glob, time
import pandas as pd
import numpy as np

# Datatypes holding dates rather than numeric values
DATE_DATATYPES = {'WC05350'}

# Patterns for extracting the delisting date from the series name
DELIST_PATS = [
    r'DELIST\.?\s*(\d{1,2}/\d{1,2}/\d{2,4})',
    r'\bDEL\.?\s*(\d{1,2}/\d{1,2}/\d{2,4})',
    r'DEAD\s*-\s*(\d{1,2}/\d{1,2}/\d{2,4})',
    r'MERGER\s*(\d{1,2}/\d{1,2}/\d{2,4})',
    r'(\d{1,2}/\d{1,2}/\d{2,4})',
]


def parse_delist_date(s):
    """'28/05/04' -> Timestamp. Two-digit years pivot at 1970."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return pd.NaT
    try:
        d, m, y = str(s).split('/')
        y = int(y)
        if y < 100:
            y = 2000 + y if y < 70 else 1900 + y
        return pd.Timestamp(year=y, month=int(m), day=int(d))
    except Exception:
        return pd.NaT


def _extract_meta_row(sym, name):
    up = name.upper()
    is_dead = 'DEAD' in up
    dd = None
    if is_dead or 'MERGER' in up or 'DELIST' in up:
        for p in DELIST_PATS:
            mm = re.search(p, name)
            if mm:
                dd = mm.group(1)
                break
    return {'symbol': sym, 'name_raw': name, 'is_dead': is_dead, 'delist_str': dd}


def parse_ds_file(path):
    """
    Parse one Datastream Excel export (exported with 'Display Code' enabled).

    Returns
    -------
    values   : long DataFrame (symbol, date, datatype, value)      - numeric series
    dates_df : long DataFrame (symbol, date, datatype, value_date) - date series (WC05350)
    meta_raw : DataFrame (symbol, name_raw, is_dead, delist_str)   - one row per column
    n_err    : number of #ERROR columns
    """
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    ws = wb[wb.sheetnames[0]]
    rows = list(ws.iter_rows(values_only=True))
    wb.close()

    ci = next((i for i, r in enumerate(rows[:10]) if r and r[0] == 'Code'), None)
    if ci is None:
        raise ValueError(f"'Code' row not found in {os.path.basename(path)}")

    code_row, name_row, data_rows = rows[ci], rows[ci - 1], rows[ci + 1:]
    dates = [r[0] for r in data_rows if r and r[0] is not None]
    n = len(dates)

    colmap, n_err = {}, 0
    for j, cd in enumerate(code_row):
        if j == 0:
            continue
        if cd is None:
            if j < len(name_row) and name_row[j] and '#ERROR' in str(name_row[j]):
                n_err += 1
            continue
        m = re.match(r'^(.+?)\(([^)]+)\)$', str(cd))
        if m:
            colmap[j] = (m.group(1).strip(), m.group(2).strip())
        else:
            colmap[j] = (str(cd).strip(), 'X')

    num_parts, date_parts, meta_rows = [], [], []
    for j, (sym, dt) in colmap.items():
        vals = [r[j] if j < len(r) else None for r in data_rows[:n]]

        if dt in DATE_DATATYPES:
            s = pd.Series(vals)
            s = s.where(s.apply(lambda x: hasattr(x, 'year')))
            if s.notna().sum() > 0:
                date_parts.append(pd.DataFrame({'symbol': sym, 'date': dates,
                                                'datatype': dt, 'value_date': s.values}))
        else:
            s = pd.to_numeric(pd.Series(vals).replace('NA', np.nan), errors='coerce')
            if s.notna().sum() > 0:
                num_parts.append(pd.DataFrame({'symbol': sym, 'date': dates,
                                               'datatype': dt, 'value': s.values}))

        nm = str(name_row[j]) if j < len(name_row) and name_row[j] else ''
        meta_rows.append(_extract_meta_row(sym, nm))

    values = (pd.concat(num_parts, ignore_index=True).dropna(subset=['value'])
              if num_parts else pd.DataFrame(columns=['symbol', 'date', 'datatype', 'value']))
    dates_df = (pd.concat(date_parts, ignore_index=True).dropna(subset=['value_date'])
                if date_parts else pd.DataFrame(columns=['symbol', 'date', 'datatype', 'value_date']))
    meta_raw = pd.DataFrame(meta_rows)
    return values, dates_df, meta_raw, n_err


def aggregate_meta(meta_raw):
    """
    Collapse per-column metadata to one row per security.

    A security appears in several files and, within each file, in several columns.
    Market series carry the 'DEAD' marker and often the delisting date in their name;
    Worldscope columns carry the accounting item name instead. Aggregation must
    therefore take the union of the evidence, not the first row encountered.
    """
    meta_raw = meta_raw.copy()
    meta_raw['delist_date'] = meta_raw['delist_str'].apply(parse_delist_date)
    g = meta_raw.groupby('symbol')

    is_dead = g['is_dead'].any()
    dead_names = meta_raw[meta_raw['is_dead']].groupby('symbol')['name_raw'].first()
    any_names = g['name_raw'].first()
    name_raw = dead_names.reindex(any_names.index).fillna(any_names)
    delist_date = g['delist_date'].min()

    return pd.DataFrame({'name_raw': name_raw,
                         'is_dead': is_dead,
                         'delist_date': delist_date}).reset_index()


def load_all(data_dir, pattern='MKT_*.xlsx', verbose=True):
    """Read every block file and consolidate by frequency family."""
    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    fam_vals = {'M': [], 'W': [], 'A': []}
    fam_dates, metas, log = [], [], []

    for f in files:
        base = os.path.basename(f)
        parts = base.replace('.xlsx', '').split('_')
        if len(parts) < 3 or parts[1] not in fam_vals:
            print(f"  [SKIP] unrecognised filename: {base}")
            continue
        fam, blk = parts[1], parts[2]

        t = time.time()
        try:
            v, d, md, nerr = parse_ds_file(f)
        except Exception as e:
            print(f"  [ERROR] {base}: {e}")
            log.append({'file': base, 'freq': fam, 'block': blk, 'rows': 0, 'symbols': 0,
                        'err_cols': None, 'sec': 0, 'status': f'FAIL: {e}'})
            continue

        v['block'] = blk
        fam_vals[fam].append(v)
        if len(d):
            d['block'] = blk
            fam_dates.append(d)
        metas.append(md)

        log.append({'file': base, 'freq': fam, 'block': blk, 'rows': len(v),
                    'symbols': v['symbol'].nunique(), 'err_cols': nerr,
                    'sec': round(time.time() - t, 1), 'status': 'OK'})
        if verbose:
            print(f"  {base:24s} {len(v):>10,} rows  {v['symbol'].nunique():>4} sec  "
                  f"{nerr:>5} #ERR  {time.time()-t:>5.1f}s")

    out = {fam: (pd.concat(fam_vals[fam], ignore_index=True) if fam_vals[fam] else pd.DataFrame())
           for fam in ['M', 'W', 'A']}
    fy = pd.concat(fam_dates, ignore_index=True) if fam_dates else pd.DataFrame()
    meta = aggregate_meta(pd.concat(metas, ignore_index=True)) if metas else pd.DataFrame()

    return out, fy, meta, pd.DataFrame(log)


## 3. Read all blocks

Roughly two seconds per file.

In [16]:
out, fy, meta, log = load_all(DATA_DIR, pattern="MKT_*.xlsx", verbose=True)

/tmp/ipykernel_1562/2706112898.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = pd.to_numeric(pd.Series(vals).replace('NA', np.nan), errors='coerce')
/tmp/ipykernel_1562/2706112898.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = pd.to_numeric(pd.Series(vals).replace('NA', np.nan), errors='coerce')
/tmp/ipykernel_1562/2706112898.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`

  MKT_A_DE-01.xlsx             14,679 rows    96 sec    961 #ERR    1.3s


/tmp/ipykernel_1562/2706112898.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = pd.to_numeric(pd.Series(vals).replace('NA', np.nan), errors='coerce')
/tmp/ipykernel_1562/2706112898.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = pd.to_numeric(pd.Series(vals).replace('NA', np.nan), errors='coerce')
/tmp/ipykernel_1562/2706112898.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`

  MKT_A_DE-02.xlsx             10,005 rows    77 sec   1273 #ERR    1.3s


/tmp/ipykernel_1562/2706112898.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = pd.to_numeric(pd.Series(vals).replace('NA', np.nan), errors='coerce')
/tmp/ipykernel_1562/2706112898.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = pd.to_numeric(pd.Series(vals).replace('NA', np.nan), errors='coerce')
/tmp/ipykernel_1562/2706112898.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`

...

  MKT_W_IT-07.xlsx            146,285 rows   148 sec      2 #ERR    2.1s


/tmp/ipykernel_1562/2706112898.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = pd.to_numeric(pd.Series(vals).replace('NA', np.nan), errors='coerce')


  MKT_W_IT-08.xlsx            153,513 rows   150 sec      0 #ERR    2.0s


/tmp/ipykernel_1562/2706112898.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = pd.to_numeric(pd.Series(vals).replace('NA', np.nan), errors='coerce')


  MKT_W_IT-09.xlsx             76,686 rows    82 sec      4 #ERR    1.1s


## 4. Summary

In [17]:
print("=== VOLUMES ===")
for fam, label in [('M', 'monthly'), ('W', 'weekly'), ('A', 'annual')]:
    d = out[fam]
    if len(d):
        print(f"{label:9s}: {len(d):>12,} rows | {d['symbol'].nunique():>6,} securities | "
              f"{d['block'].nunique():>3} blocks")
        print(f"           datatypes: {sorted(d['datatype'].unique())}")

print(f"\nFiscal year ends : {len(fy):>12,} rows | "
      f"{fy['symbol'].nunique() if len(fy) else 0:>6,} securities")
print(f"Security metadata: {len(meta):>12,} securities | "
      f"{int(meta['is_dead'].sum()):,} dead | "
      f"{int(meta['delist_date'].notna().sum()):,} with delisting date")

print("\n=== FILES WITH PROBLEMS ===")
bad = log[log['status'] != 'OK']
print(bad.to_string(index=False) if len(bad) else "  none")

print("\n=== BLOCKS WITH MOST #ERROR COLUMNS ===")
print(log.nlargest(10, 'err_cols')[['file', 'symbols', 'err_cols']].to_string(index=False))

=== VOLUMES ===
monthly  :   13,600,077 rows |  6,896 securities |  52 blocks
           datatypes: ['AF', 'DPS', 'DY', 'EPS1MN', 'EPS1SD', 'MV', 'NOSH', 'P', 'RI', 'VO']
weekly   :    7,399,302 rows |  6,350 securities |  52 blocks
           datatypes: ['RI']
annual   :    1,028,439 rows |  5,200 securities |  52 blocks
           datatypes: ['WC01001', 'WC01051', 'WC01151', 'WC01551', 'WC01751', 'WC02003', 'WC02201', 'WC02999', 'WC03051', 'WC03063', 'WC03101', 'WC03251', 'WC03501', 'WC04601', 'WC18191']

Fiscal year ends :       76,724 rows |  5,202 securities
Security metadata:        7,132 securities | 4,077 dead | 3,836 with delisting date

=== FILES WITH PROBLEMS ===
  none

=== BLOCKS WITH MOST #ERROR COLUMNS ===
            file  symbols  err_cols
MKT_A_FR-04.xlsx        6      2305
MKT_A_FR-03.xlsx       15      2193
MKT_A_FR-02.xlsx       26      2058
MKT_A_DE-05.xlsx       54      1591
MKT_M_DE-04.xlsx       91      1533
MKT_A_FR-15.xlsx       64      1452
MKT_A_FR-01.xlsx 

### Coverage by datatype

Share of securities with at least one valid observation. Sparse datatypes translate
directly into sparse characteristics downstream.

In [18]:
for fam, label in [('M', 'MONTHLY'), ('A', 'ANNUAL')]:
    d = out[fam]
    if not len(d):
        continue
    print(f"=== {label} ===")
    cov = d.groupby('datatype')['symbol'].nunique().sort_values(ascending=False)
    tot = d['symbol'].nunique()
    for dt, n in cov.items():
        print(f"  {dt:10s} {n:>6,} / {tot:,}  ({100*n/tot:>5.1f}%)")
    print()

=== MONTHLY ===
  AF          6,470 / 6,896  ( 93.8%)
  DPS         6,464 / 6,896  ( 93.7%)
  NOSH        6,443 / 6,896  ( 93.4%)
  P           6,360 / 6,896  ( 92.2%)
  DY          6,359 / 6,896  ( 92.2%)
  RI          6,344 / 6,896  ( 92.0%)
  MV          6,100 / 6,896  ( 88.5%)
  VO          5,645 / 6,896  ( 81.9%)
  EPS1MN      3,852 / 6,896  ( 55.9%)
  EPS1SD      3,209 / 6,896  ( 46.5%)

=== ANNUAL ===
  WC01001     5,193 / 5,200  ( 99.9%)
  WC01751     5,186 / 5,200  ( 99.7%)
  WC02999     5,186 / 5,200  ( 99.7%)
  WC03501     5,186 / 5,200  ( 99.7%)
  WC01551     5,185 / 5,200  ( 99.7%)
  WC03251     5,181 / 5,200  ( 99.6%)
  WC03051     5,129 / 5,200  ( 98.6%)
  WC18191     5,120 / 5,200  ( 98.5%)
  WC01151     5,060 / 5,200  ( 97.3%)
  WC04601     5,008 / 5,200  ( 96.3%)
  WC01051     4,515 / 5,200  ( 86.8%)
  WC02003     4,425 / 5,200  ( 85.1%)
  WC02201     4,242 / 5,200  ( 81.6%)
  WC03101     4,234 / 5,200  ( 81.4%)
  WC03063     3,323 / 5,200  ( 63.9%)



### Cross-check against the master universe

Compares the dead/active status inferred from the series names with the status
recorded in the Navigator extraction.

In [19]:
mp = os.path.join(DATA_DIR, "master_universe.csv")
if os.path.exists(mp):
    u = pd.read_csv(mp, dtype=str)
    m = meta.merge(u[['Symbol', 'Activity', 'country']], left_on='symbol',
                   right_on='Symbol', how='left')
    print("Navigator status (rows) vs 'DEAD' marker in series name (columns):")
    print(pd.crosstab(m['Activity'], m['is_dead'], dropna=False))

    n_dead = int((m['Activity'] == 'Dead').sum())
    n_nodate = int(((m['Activity'] == 'Dead') & m['delist_date'].isna()).sum())
    print(f"\nDelisted securities without a parsed date: {n_nodate:,} of {n_dead:,} "
          f"({100*n_nodate/max(n_dead,1):.1f}%)")
    print("For these the delisting point will be recovered by truncating the return index.")

    univ = set(u['Symbol'])
    have = set(meta['symbol'])
    print(f"\nUniverse: {len(univ):,} | present in the data: {len(have & univ):,} "
          f"({100*len(have & univ)/len(univ):.1f}%)")
    missing = univ - have
    if missing:
        d = u[u['Symbol'].isin(missing)]
        print(f"Never retrieved: {len(missing):,}  by country {d['country'].value_counts().to_dict()}")
else:
    print("universo_master.csv not found - skipping cross-check.")

Navigator status (rows) vs 'DEAD' marker in series name (columns):
is_dead   False  True 
Activity              
Active     1893      1
Dead       1162   4076

Delisted securities without a parsed date: 1,403 of 5,238 (26.8%)
For these the delisting point will be recovered by truncating the return index.

Universe: 7,529 | present in the data: 7,132 (94.7%)
Never retrieved: 397  by country {'DE': 273, 'FR': 76, 'ES': 32, 'IT': 16}


## 5. Save

In [28]:
def save(df, name):
    if not len(df):
        print(f"  [EMPTY] {name} not written")
        return
    p = os.path.join(DATA_DIR, name)
    df.to_parquet(p, index=False)
    print(f"  {name:26s} {len(df):>12,} rows  ({os.path.getsize(p)/1e6:.1f} MB)")

print("Writing:")
save(out['M'], "raw_monthly.parquet")
save(out['W'], "raw_weekly.parquet")
save(out['A'], "raw_annual.parquet")
save(fy,       "fiscal_year_end.parquet")
save(meta,     "security_meta.parquet")
log.to_csv(os.path.join(DATA_DIR, "read_log.csv"), index=False)
print(f"  {'read_log.csv':26s} {len(log):>12,} rows")

Writing:
  raw_monthly.parquet          13,600,077 rows  (16.4 MB)
  raw_weekly.parquet            7,399,302 rows  (18.3 MB)
  raw_annual.parquet            1,028,439 rows  (4.6 MB)
  fiscal_year_end.parquet          76,724 rows  (0.1 MB)
  security_meta.parquet             7,132 rows  (0.2 MB)
  read_log.csv                        156 rows


## 6. Standalone series: risk-free rate and market indices

Same file structure, but one series per column rather than a block of securities.
Adjust the filenames below if they differ.

In [30]:
for fname, outname in [("RF_M.xlsx", "raw_riskfree.parquet"),
                       ("IDX_W.xlsx", "raw_index.parquet")]:
    p = os.path.join(DATA_DIR, fname)
    if not os.path.exists(p):
        print(f"[MISSING] {fname} - skipped")
        continue
    v, d, md, nerr = parse_ds_file(p)
    print(f"{fname}: {len(v):,} rows | series: {sorted(v['symbol'].unique())} | #ERROR {nerr}")
    if len(v):
        print(f"  period: {v['date'].min()} -> {v['date'].max()}")
        v.to_parquet(os.path.join(DATA_DIR, outname), index=False)
        print(f"  written to {outname}")

RF_M.xlsx: 360 rows | series: ['EIBOR3M'] | #ERROR 0
  period: 1996-01-01 00:00:00 -> 2025-12-01 00:00:00
  written to raw_riskfree.parquet
IDX_W.xlsx: 7,830 rows | series: ['TOTMKBD', 'TOTMKES', 'TOTMKEU', 'TOTMKFR', 'TOTMKIT'] | #ERROR 0
  period: 1996-01-01 00:00:00 -> 2025-12-29 00:00:00
  written to raw_index.parquet


## 7. Verification

Reload the saved files to confirm they are readable and complete.

In [31]:
for f in ["raw_monthly.parquet", "raw_weekly.parquet", "raw_annual.parquet",
          "fiscal_year_end.parquet", "security_meta.parquet",
          "raw_riskfree.parquet", "raw_index.parquet"]:
    p = os.path.join(DATA_DIR, f)
    if os.path.exists(p):
        d = pd.read_parquet(p)
        print(f"{f:26s} OK  {len(d):>12,} rows x {d.shape[1]} cols")
    else:
        print(f"{f:26s} not present")

raw_monthly.parquet        OK    13,600,077 rows x 5 cols
raw_weekly.parquet         OK     7,399,302 rows x 5 cols
raw_annual.parquet         OK     1,028,439 rows x 5 cols
fiscal_year_end.parquet    OK        76,724 rows x 5 cols
security_meta.parquet      OK         7,132 rows x 4 cols
raw_riskfree.parquet       OK           360 rows x 4 cols
raw_index.parquet          OK         7,830 rows x 4 cols


In [32]:
u = pd.read_csv(os.path.join(DATA_DIR, "master_universe.csv"), dtype=str)
meta = pd.read_parquet(os.path.join(DATA_DIR, "security_meta.parquet"))
missing = u[~u['Symbol'].isin(set(meta['symbol']))]
print(missing['block'].value_counts().head(10))

block
DE-03    52
DE-04    32
DE-06    20
DE-05    20
DE-15    17
FR-01    17
DE-14    17
DE-12    16
DE-01    16
DE-10    15
Name: count, dtype: int64
